<a href="https://colab.research.google.com/github/TheGreatWall32/IPL/blob/main/Session2_DecisionTrees_Carseats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Trees: Classification with Carseats Dataset

## Session 2 - Part 2

In this notebook, you will:
1. Build and evaluate classification trees
2. Explore the effects of hyperparameters
3. Analyze overfitting and model selection
4. Compare single trees with ensemble methods

**Dataset**: Carseats from ISLP - simulated data on child car seat sales at 400 stores.

**Target**: Predict whether a store has high sales (Sales > 8).

**Business Context**:
The Carseats dataset is a simulated dataset that represents sales data for child car seats at 400 different retail stores. It's designed to explore what factors influence whether a store achieves high or low sales performance.
Think of it as a retail analytics problem: a company sells child car seats through various stores and wants to understand what drives sales success.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)

---
## Data Loading

1. Install ISLP
2. Load the Carseats dataset
3. Explore the data

In [ ]:
!pip install ISLP

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.4 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=b4daf5f96d461ae43ff0bb8b7600c6bc0539c7499f69ac020ea2c329a67290c3
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [ ]:
# Load the Carseats dataset
try:
    from ISLP import load_data
    carseats = load_data('Carseats')
    print("Loaded Carseats from ISLP")
except ImportError:
    carseats = None
    print("ISLP not available - install using:\n pip install ISLP")

if carseats is not None:
    print(f"Shape: {carseats.shape}")
    display(carseats.head())

Loaded Carseats from ISLP
Shape: (400, 11)


,Sales,CompPrice,Income,Advertising,Population,Price,ShelveLoc,Age,Education,Urban,US
0,9.50,138,73,11,276,120,Bad,42,17,Yes,Yes
1,11.22,111,48,16,260,83,Good,65,10,Yes,Yes
2,10.06,113,35,10,269,80,Medium,59,12,Yes,Yes
3,7.40,117,100,4,466,97,Medium,55,14,Yes,Yes
4,4.15,141,64,3,340,128,Bad,38,13,Yes,No


In [ ]:
# Explore the data
carseats.info()
print("\n")
carseats.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   Sales        400 non-null    float64 
 1   CompPrice    400 non-null    int64   
 2   Income       400 non-null    int64   
 3   Advertising  400 non-null    int64   
 4   Population   400 non-null    int64   
 5   Price        400 non-null    int64   
 6   ShelveLoc    400 non-null    category
 7   Age          400 non-null    int64   
 8   Education    400 non-null    int64   
 9   Urban        400 non-null    category
 10  US           400 non-null    category
dtypes: category(3), float64(1), int64(7)
memory usage: 26.7 KB




,Sales,CompPrice,Income,Advertising,Population,Price,Age,Education
count,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000
mean,7.496325,124.975000,68.657500,6.635000,264.840000,115.795000,53.322500,13.900000
std,2.824115,15.334512,27.986037,6.650364,147.376436,23.676664,16.200297,2.620528
min,0.000000,77.000000,21.000000,0.000000,10.000000,24.000000,25.000000,10.000000
25%,5.390000,115.000000,42.750000,0.000000,139.000000,100.000000,39.750000,12.000000
50%,7.490000,125.000000,69.000000,5.000000,272.000000,117.000000,54.500000,14.000000
75%,9.320000,135.000000,91.000000,12.000000,398.500000,131.000000,66.000000,16.000000
max,16.270000,175.000000,120.000000,29.000000,509.000000,191.000000,80.000000,18.000000


---
## Data Preparation

1. Create a binary target variable `High` where `High = 'Yes'` if `Sales > 8`, else `'No'`

   = Create a new column `High` that equals `'Yes'` if Sales > 8, otherwise `'No'`.
  
   **Hint**: Use `np.where(condition, value_if_true, value_if_false)`
   
   
2. Identify categorical columns and apply one-hot encoding
3. Create feature matrix `X` (exclude `Sales` and `High`) and target vector `y`
4. Split into 70% training, 30% test sets

In [ ]:
# YOUR CODE HERE

# 1. Create binary target

# Create the 'High' column
carseats['High'] = ___  # Fill in the blank
print(f"Target distribution:\n{carseats['High'].value_counts()}")


Target distribution:
High
    400
Name: count, dtype: int64


---
## Data Preparation

2. Identify categorical columns and apply one-hot encoding


In [ ]:
# 2. One-hot encode categorical columns
cat_cols = ['']    # Which columns to encode?
carseats_encoded = pd.get_dummies(carseats, columns=cat_cols, drop_first=True)


---
## Data Preparation

3. Create feature matrix `X` (exclude `?` and `?`) and target vector `y`

In [ ]:
# 3. Create X and y
# Define feature columns (everything except ???)
feature_cols = [col for col in carseats_encoded.columns if col not in ['', '']]   # What are not features?
print(f"Features: {feature_cols}")

# Create X (features) and y (target - what is it?)
X = carseats_encoded[feature_cols]
y = (carseats[''] == 'Yes').astype(int)  # Convert to 0/1

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")



---
## Data Preparation

4. Split into 70% training, 30% test sets


In [ ]:
# 4. Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    ___,  # Features
    ___,  # Target
    test_size=___,  # What fraction for testing?
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nTraining set - High sales: {y_train.mean():.1%}")
print(f"Test set - High sales: {y_test.mean():.1%}")

---
## Quick Recap

Last class, we learned:
- **Entropy** measures impurity: $-\sum p_k \log_2(p_k)$
- **Gini Index** also measures impurity: $1 - \sum p_k^2$
- **Information Gain** = Parent entropy - Weighted child entropy
- Trees split on the feature with **highest information gain**


---
## 1) Build a Classification Tree

1. Create a `DecisionTreeClassifier` with `criterion='entropy'` and `max_depth=4`
2. Fit on training data
3. Make predictions
4. Report training and test accuracy
5. Display the confusion matrix for test predictions
4. Visualize the tree using `plot_tree()`
5. Print the decision rules using `export_text()`

In [ ]:
# Your code here

# Step 1: Create the classifier
clf = DecisionTreeClassifier(
    criterion=___,
    max_depth=___,
    min_samples_leaf=___,
    random_state=42
)

# Step 2: Fit on training data
clf.fit(___, ___)

# Step 3: Make predictions
y_train_pred = clf.predict(___)
y_test_pred = clf.predict(___)

# Step 4: Calculate accuracy
train_acc = accuracy_score(___, ___)
test_acc = accuracy_score(___, ___)

print("Decision Tree Results")
print("=" * 40)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:     {test_acc:.4f}")
print(f"\nTree Depth: {clf.get_depth()}")
print(f"Number of Leaves: {clf.get_n_leaves()}")

# Step 5: Confusion matrix
cm = confusion_matrix(___, ___)
print("Confusion Matrix (Test Set)")
print("=" * 35)
print(f"             Predicted")
print(f"           Low    High")
print(f"Actual Low  {cm[0,0]:3d}    {cm[0,1]:3d}")
print(f"Actual High {cm[1,0]:3d}    {cm[1,1]:3d}")
print(f"\nTrue Negatives (correct Low):   {cm[0,0]}")
print(f"False Positives (wrong High):   {cm[0,1]}")
print(f"False Negatives (wrong Low):    {cm[1,0]}")
print(f"True Positives (correct High):  {cm[1,1]}")
print(f"\nClassification Report:\n{classification_report(y_test, clf.predict(X_test))}")

# Step 6: Visualize the tree
fig, ax = plt.subplots(figsize=(20, 12))

plot_tree(
    ___,  # Your trained classifier
    feature_names=feature_cols,
    class_names=['Low', 'High'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)

plt.title('Classification Tree for High Sales', fontsize=16)
plt.tight_layout()
plt.show()

# Step 7: Export the decision rules:
print(export_text(
    ___,   # Your trained classifier
    feature_names=feature_cols))

---
## 2) Hyperparameter Analysis

1. Train trees with `max_depth` ranging from 1 to 20
2. Plot training accuracy vs test accuracy as a function of depth
3. Identify the optimal depth based on test accuracy
4. At what depth does overfitting become evident?
5. Test it with Gini index criterion. Can you see a difference?

**Bonus**:     
   1) Repeat the analysis for `min_samples_leaf` values [1, 5, 10, 20, 50] with unlimited depth.

In [ ]:
# Depth analysis
depths = range(1, 21)
train_accs = []
test_accs = []

for depth in depths:
    # Build a tree with this depth
    tree = DecisionTreeClassifier(
        criterion='entropy',
        max_depth=___,  # loop variable
        random_state=42
    )

    # Fit and evaluate
    tree.fit(___, ___)

    train_accs.append(accuracy_score(___, tree.predict(___)))
    test_accs.append(accuracy_score(___, tree.predict(___)))

# Find best depth
best_depth = depths[np.argmax(test_accs)]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(depths, train_accs, 'b-o', label='Training', linewidth=2)
ax.plot(depths, test_accs, 'r-s', label='Test', linewidth=2)
ax.axvline(best_depth, color='green', linestyle='--', label=f'Best: {best_depth}')
ax.set_xlabel('Tree Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: Training vs Test Accuracy')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(f"\nOptimal depth: {best_depth}")
print(f"Best test accuracy: {max(test_accs):.4f}")

---
## Optional Exercise: Feature Importance

1. Extract feature importances from your best tree
2. Create a horizontal bar plot showing importance of each feature
3. Which features are most predictive of high sales? Does this make business sense?

In [ ]:
# Your code here


---
## Optional Exercise: Gini vs Entropy

Compare trees built with `criterion='gini'` vs `criterion='entropy'`.

1. Train both with optimal depth from Exercise 4
2. Compare test accuracies
3. Compare the resulting tree structures (depth, number of leaves)
4. Do they make the same predictions? Calculate agreement percentage.

In [ ]:
# Your code here


---
## Optional Exercise: Prediction Analysis

1. Find observations where your tree predicts incorrectly on the test set
2. Analyze these misclassified cases - are there patterns?
3. Use `predict_proba()` to get prediction confidence
4. Are errors associated with low-confidence predictions?

In [ ]:
# Your code here


**Disclosure**: *Portions of these educational materials were prepared with assistance from AI assisstent (Claude, Anthropic). The author reviewed, edited, and verified all AI-generated content.*